# Lab: BiLSTM Sentiment Classifier on SST-2
Цей зошит містить повну реалізацію лабораторної роботи з класифікації тональності тексту на датасеті SST-2 за допомогою двонаправленої LSTM (BiLSTM) на PyTorch.

## Part 1 - Setup & Data Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import numpy as np
import pandas as pd
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset = load_dataset("glue", "sst2")
print(f"Train size: {len(dataset['train'])}")
print(f"Val size : {len(dataset['validation'])}")
print(f"\nSample:\n{dataset['train'][0]}")

Using device: cpu


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Train size: 67349
Val size : 872

Sample:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [ ]:
train_df = pd.DataFrame(dataset["train"])

print("Label counts:")
print(train_df["label"].value_counts())
print(f"\nPositive ratio: {train_df['label'].mean():.2%}")

train_df["num_words"] = train_df["sentence"].apply(lambda s: len(s.split()))
print(f"\nSentence length stats:")
print(train_df["num_words"].describe())

Label counts:
label
1    37569
0    29780
Name: count, dtype: int64

Positive ratio: 55.78%

Sentence length stats:
count    67349.000000
mean         9.409553
std          8.073806
min          1.000000
25%          3.000000
50%          7.000000
75%         13.000000
max         52.000000
Name: num_words, dtype: float64


## Part 2 - Tokenization & Vocabulary

In [3]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_IDX = 0
UNK_IDX = 1

def tokenize(sentence):
    """Simple lowercase whitespace tokenizer."""
    return sentence.lower().split()

def build_vocab(dataset_split, min_freq=2):
    """Build word-to-index mapping from training data."""
    counter = Counter()
    for example in dataset_split:
        tokens = tokenize(example["sentence"])
        counter.update(tokens)
        
    word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    idx = 2
    for word, count in counter.items():
        if count >= min_freq:
            word2idx[word] = idx
            idx += 1
            
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word

word2idx, idx2word = build_vocab(dataset["train"], min_freq=2)
V = len(word2idx)
print(f"Vocabulary size: {V}")
print(f"Sample: {dict(list(word2idx.items())[2:7])}")

Vocabulary size: 14311
Sample: {'hide': 2, 'new': 3, 'secretions': 4, 'from': 5, 'the': 6}


In [4]:
MAX_LEN = 50

def encode_sentence(sentence, word2idx, max_len=MAX_LEN):
    """Convert sentence string to padded list of token IDs."""
    tokens = tokenize(sentence)
    ids = [word2idx.get(t, UNK_IDX) for t in tokens]
    # Truncate
    ids = ids[:max_len]
    # Pad
    pad_length = max_len - len(ids)
    ids = ids + [PAD_IDX] * pad_length
    return ids

class SST2Dataset(Dataset):
    """PyTorch Dataset wrapper for SST-2."""
    def __init__(self, hf_split, word2idx, max_len=MAX_LEN):
        self.data = hf_split
        self.word2idx = word2idx
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        example = self.data[idx]
        input_ids = encode_sentence(example["sentence"], self.word2idx, self.max_len)
        label = example["label"]
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

BATCH_SIZE = 64
train_dataset = SST2Dataset(dataset["train"], word2idx)
val_dataset = SST2Dataset(dataset["validation"], word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch_x, batch_y = next(iter(train_loader))
print(f"Batch input shape : {batch_x.shape}")
print(f"Batch label shape : {batch_y.shape}")

Batch input shape : torch.Size([64, 50])
Batch label shape : torch.Size([64])


## Part 3 - Implement BiLSTM Model

In [5]:
class BiLSTMSentiment(nn.Module):
    """
    Bidirectional LSTM for binary sentiment classification.
    Architecture: Embedding -> BiLSTM -> Dropout -> FC -> logits
    """
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers, num_classes=2, dropout=0.3, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded)
        h_forward = hidden[-2, :, :]
        h_backward = hidden[-1, :, :]
        h_cat = torch.cat([h_forward, h_backward], dim=1)
        logits = self.fc(self.dropout(h_cat))
        return logits

EMB_DIM = 100
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
NUM_CLASSES = 2

model = BiLSTMSentiment(
    vocab_size=V,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Total parameters : 2,062,398
Trainable parameters: 2,062,398


## Part 4 - Training & Evaluation

In [6]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch, return average loss and accuracy."""
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == batch_y).sum().item()
        total_examples += batch_x.size(0)
        
    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    """Evaluate model, return average loss and accuracy."""
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            total_loss += loss.item() * batch_x.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == batch_y).sum().item()
            total_examples += batch_x.size(0)
            
    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    return avg_loss, accuracy

NUM_EPOCHS = 5
print(f"{'Epoch':>5} {'Train Loss':>10} {'Train Acc':>9} {'Val Loss':>10} {'Val Acc':>9}")
print("-" * 55)
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"{epoch:5d} {train_loss:10.4f} {train_acc:8.2%} {val_loss:10.4f} {val_acc:8.2%}")

Epoch Train Loss Train Acc   Val Loss   Val Acc
-------------------------------------------------------
    1     0.5495   70.68%     0.4869   78.21%
    2     0.3620   83.78%     0.4485   82.00%
    3     0.2781   88.31%     0.4451   83.37%
    4     0.2334   90.54%     0.4752   83.14%
    5     0.1992   92.01%     0.4722   83.83%


## Part 5 - Inspect Predictions

In [7]:
def predict_sentiment(model, sentence, word2idx, device):
    """Predict sentiment for a single sentence."""
    model.eval()
    input_ids = encode_sentence(sentence, word2idx)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()
        conf = probs[0, pred].item()
        
    label_str = "Positive" if pred == 1 else "Negative"
    return label_str, conf

test_sentences = [
    "This movie was absolutely wonderful and deeply moving.",
    "A total waste of time, boring and predictable.",
    "Not great not terrible, just okay.",
    "Although the plot was weak, the acting was superb.",
    "I hated every minute of this dreadful film.",
    "An unexpectedly delightful surprise."
]

print(f"{'Prediction':>10} {'Conf':>6} Sentence")
print("-" * 70)
for sent in test_sentences:
    label, conf = predict_sentiment(model, sent, word2idx, device)
    print(f"{label:>10} {conf:5.1%} {sent}")

Prediction   Conf Sentence
----------------------------------------------------------------------
  Positive 99.9% This movie was absolutely wonderful and deeply moving.
  Negative 99.8% A total waste of time, boring and predictable.
  Negative 54.2% Not great not terrible, just okay.
  Negative 79.0% Although the plot was weak, the acting was superb.
  Negative 98.8% I hated every minute of this dreadful film.
  Positive 99.9% An unexpectedly delightful surprise.


In [8]:
def get_errors(model, hf_split, word2idx, device, max_errors=10):
    """Find misclassified examples from a dataset split."""
    model.eval()
    errors = []
    for example in hf_split:
        sentence = example["sentence"]
        true_label = example["label"]
        
        pred_label, conf = predict_sentiment(model, sentence, word2idx, device)
        pred_int = 1 if pred_label == "Positive" else 0
        
        if pred_int != true_label:
            errors.append({
                "sentence": sentence,
                "true": "Pos" if true_label == 1 else "Neg",
                "pred": pred_label,
                "conf": conf,
            })
            if len(errors) >= max_errors:
                break
    return errors

errors = get_errors(model, dataset["validation"], word2idx, device, max_errors=10)
print(f"\n--- First {len(errors)} misclassified validation examples ---\n")
for e in errors:
    print(f"True: {e['true']:>3} | Pred: {e['pred']:>8} ({e['conf']:.1%}) | {e['sentence']}")


--- First 10 misclassified validation examples ---

True: Pos | Pred: Negative (67.1%) | we root for ( clara and paul ) , even like them , though perhaps it 's an emotion closer to pity . 
True: Neg | Pred: Positive (97.9%) | pumpkin takes an admirable look at the hypocrisy of political correctness , but it does so with such an uneven tone that you never know when humor ends and tragedy begins . 
True: Neg | Pred: Positive (85.6%) | the iditarod lasts for days - this just felt like it did . 
True: Neg | Pred: Positive (93.9%) | it 's a cookie-cutter movie , a cut-and-paste job . 
True: Pos | Pred: Negative (86.5%) | escaping the studio , piccoli is warmly affecting and so is this adroitly minimalist movie . 
True: Neg | Pred: Positive (62.9%) | the title not only describes its main characters , but the lazy people behind the camera as well . 
True: Neg | Pred: Positive (98.8%) | it offers little beyond the momentary joys of pretty and weightless intellectual entertainment . 
True: Pos